# GKC Authentication Quick Start

This notebook mirrors the Authentication API quick start and examples for `gkc.auth`.

It is designed for two modes:

- **Offline checks** for constructor behavior, endpoint resolution, and helper methods
- **Optional live checks** against Wikimedia APIs when credentials are available

## Imports

Import the authentication classes and constants used in the examples.

In [1]:
from gkc import AuthenticationError, WikiverseAuth

## Managing and Using Credentials

To conduct write operations on partner systems (Wikimedia projects, OpenStreetMap, etc.), you will need to supply credentials. The general modality for gkc operations was designed initially for bot account types of activity. Future development may incorporate OAuth routes.

### Best Practice: Use Environment Variables

The gkc package supports the use of environment variables for credentials as a best practice. This allows the package to be operated in various workflows, including use of the GKC Wizard - a locally run Streamlit app for managing curation packets, filling data, and submitting to Wikidata (and eventually other systems).

The following environment variables should be set, preferrably with a "bot password" set up under a Wikimedia account approved for bot operations.

- `WIKIVERSE_USERNAME`
- `WIKIVERSE_PASSWORD`

In [ ]:
# The cell safely skips if credentials are not present.

import os

username = os.getenv("WIKIVERSE_USERNAME")
password = os.getenv("WIKIVERSE_PASSWORD")

if not (username and password):
    print("Skipping live test: set WIKIVERSE_USERNAME and WIKIVERSE_PASSWORD to run.")
else:
    auth = WikiverseAuth()
    print("Target API:", auth.api_url)

    try:
        auth.login()
        print("Logged in:", auth.is_logged_in())

        diagnostics = auth.test_authentication()
        print("Diagnostics:")
        for key, value in diagnostics.items():
            print(f"  {key}: {value}")

        userinfo = auth.session.get(
            auth.api_url,
            params={"action": "query", "meta": "userinfo", "format": "json"},
        ).json()
        print("User from API:", userinfo.get("query", {}).get("userinfo", {}).get("name"))

        csrf_token = auth.get_csrf_token()
        print("CSRF token prefix:", csrf_token[:20] + "...")
    finally:
        auth.logout()
        print("Logged out:", not auth.is_logged_in())

### Interactive credential input

If you need to supply credentials at runtime rather than the best practice method with environment variables, you can use the following pattern to collect credentials for a given session within a notebook or similar environment.

In [ ]:
import getpass

username = input("Enter Wikiverse username (format: Username@BotName): ").strip()
password = getpass.getpass("Enter Wikiverse password: ").strip()

auth = WikiverseAuth(username=username, password=password)
print(auth)
print("Account name:", auth.get_account_name())
print("Bot name:", auth.get_bot_name())

## Endpoint Shortcuts

API URL shortcuts exist in the gkc package to help in making authenticated and non-authenticated connections to partner systems.

In [ ]:
from gkc.auth import DEFAULT_WIKIMEDIA_APIS

for shortcut in ["wikidata", "wikidata_test", "wikipedia", "commons", "datadistillery_wikibase"]:
    print(f"{shortcut:14s} -> {DEFAULT_WIKIMEDIA_APIS[shortcut]}")

## Error Handling: Missing Credentials

Demonstrate the expected `AuthenticationError` when `login()` is called without credentials.

In [ ]:
auth = WikiverseAuth(
    username="Alice@MyBot",
    password="secret",
)

try:
    auth.login()
except AuthenticationError as exc:
    print(type(exc).__name__)
    print(exc)

## Next Steps

Use this notebook as the pattern for module-level quick-start notebooks:

1. Start with imports and constructor behavior
2. Include deterministic offline checks
3. Include guarded live checks for networked/authenticated paths
4. Keep examples aligned with API documentation